# Паттерны для собеседований


Маленькие приёмы, которые позволяют решать целые классы задач за `O(n)` вместо `O(n²)`. Их часто ждут на собесе.


## Два указателя (Two Pointers)

Два индекса идут навстречу друг другу или в одну сторону с разной скоростью.

Применения: палиндром, удаление дубликатов из отсортированного массива, сумма двух чисел в отсортированном массиве, контейнер с водой.


### Проверка палиндрома


In [ ]:
def is_palindrome(s):
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return False
        left += 1
        right -= 1
    return True

print(is_palindrome('racecar'))   # True
print(is_palindrome('hello'))     # False

### Two Sum для отсортированного массива (O(n))


In [ ]:
def two_sum_sorted(arr, target):
    left, right = 0, len(arr) - 1
    while left < right:
        s = arr[left] + arr[right]
        if s == target:
            return [left, right]
        elif s < target:
            left += 1
        else:
            right -= 1
    return None

two_sum_sorted([1, 2, 4, 7, 11, 15], 9)   # [1, 3]

### Удаление дубликатов из отсортированного массива на месте


In [ ]:
def remove_duplicates(arr):
    if not arr: return 0
    slow = 0
    for fast in range(1, len(arr)):
        if arr[fast] != arr[slow]:
            slow += 1
            arr[slow] = arr[fast]
    return slow + 1   # длина уникальной части

arr = [1, 1, 2, 2, 3, 4, 4, 5]
k = remove_duplicates(arr)
print(k, arr[:k])   # 5 [1, 2, 3, 4, 5]

## Хеш-таблица / словарь (Hash Map)

Самый универсальный паттерн на собесе. Идея: пока идёшь по массиву **один раз**, запоминай в словаре полезную информацию о уже виденных элементах. На каждом шаге проверяй словарь за `O(1)` — и весь алгоритм получается `O(n)` вместо `O(n²)`.

Платим за это **дополнительной памятью** `O(n)` — почти всегда выгодная сделка.

Типовые роли словаря:
- **значение → индекс** (для Two Sum и подобных задач на «дополнение»);
- **значение → счётчик** (частоты, поиск самого частого, anagram check);
- **ключ-категория → список** (группировка анаграмм, bucket-задачи);
- **множество посещённых** (`set`) — детект дубликатов, циклов.


### Паттерн «дополнение» (Two Sum)

Найти два числа в массиве с суммой `target`. Наивно — два цикла, `O(n²)`. Через словарь — один проход, `O(n)`.

**Идея:** идём по массиву, для каждого `num` ищем «дополнение» `target - num` среди уже виденных. Если оно там — нашли пару. Иначе кладём `num` в словарь и идём дальше.


In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}                          # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

two_sum([2, 7, 11, 15], 9)   # [0, 1]

Тот же приём работает для целой россыпи задач:
- найти пару с разностью `k` → ищи `num - k` или `num + k`;
- проверить, что массив содержит **обе** копии числа (`x` и `-x`);
- посчитать пары с суммой кратной `m` → ключ = `num % m`.


### Паттерн «подсчёт частот»

Сколько раз встречается каждый элемент. Базовая операция для **anagram check**, **majority element**, **top-k frequent elements**.


In [ ]:
from collections import Counter

# Самый частый элемент
def most_common(arr):
    return Counter(arr).most_common(1)[0][0]

most_common([1, 2, 3, 2, 2, 4, 5, 2])   # 2

In [ ]:
# Анаграммы: одинаковые буквы в одинаковом количестве
def is_anagram(a: str, b: str) -> bool:
    return Counter(a) == Counter(b)

print(is_anagram('listen', 'silent'))   # True
print(is_anagram('hello', 'world'))     # False

In [ ]:
# Top-K частых элементов
from collections import Counter

def top_k_frequent(arr, k):
    return [x for x, _ in Counter(arr).most_common(k)]

top_k_frequent([1, 1, 1, 2, 2, 3], 2)   # [1, 2]

### Паттерн «запомнить последнюю позицию»

Когда нужна не просто пара, а **расстояние** между повторами или ближайшее вхождение.
В словаре храним `value → последний индекс`.


In [ ]:
# Есть ли два одинаковых числа на расстоянии <= k?
def contains_nearby_duplicate(arr, k):
    last = {}                          # value → последний индекс
    for i, x in enumerate(arr):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False

print(contains_nearby_duplicate([1, 2, 3, 1], 3))     # True
print(contains_nearby_duplicate([1, 0, 1, 1], 1))     # True
print(contains_nearby_duplicate([1, 2, 3, 1, 2], 2))  # False

### Паттерн «группировка по ключу» (defaultdict)

Когда нужно разложить элементы по корзинам по общему свойству.
Ключ — то, по чему группируем. Значение — список/множество элементов.


In [ ]:
from collections import defaultdict

# Группировка анаграмм: ключ — отсортированные буквы
def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        key = ''.join(sorted(w))
        groups[key].append(w)
    return list(groups.values())

group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat'])
# [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

In [ ]:
# Группировка чисел по остатку от деления (классический трюк)
from collections import defaultdict

def group_by_mod(arr, m):
    groups = defaultdict(list)
    for x in arr:
        groups[x % m].append(x)
    return dict(groups)

group_by_mod([1, 2, 3, 4, 5, 6, 7, 8], 3)
# {1: [1, 4, 7], 2: [2, 5, 8], 0: [3, 6]}

### Паттерн «множество посещённых» (set)

Когда не важно **сколько** раз встречали элемент, важен только сам факт. `set` даёт `O(1)` `in`-проверку.


In [ ]:
# Поиск дубликата за один проход
def has_duplicate(arr):
    seen = set()
    for x in arr:
        if x in seen:
            return True
        seen.add(x)
    return False

has_duplicate([1, 2, 3, 4, 2])   # True

In [ ]:
# Длиннейшая последовательная подпоследовательность за O(n)
# (Longest Consecutive Sequence — классика LeetCode)
def longest_consecutive(arr):
    nums = set(arr)                      # O(1) lookup
    best = 0
    for x in nums:
        if x - 1 in nums:
            continue                     # x не начало стрика — пропускаем
        length = 1
        while x + length in nums:        # тянем вправо
            length += 1
        best = max(best, length)
    return best

longest_consecutive([100, 4, 200, 1, 3, 2])   # 4 ([1, 2, 3, 4])

### Когда выбирать паттерн «словарь»

Если в задаче:
- спрашивают про **пары/тройки** с условием на сумму/разность/произведение;
- надо **найти дубликат** или сравнить вхождения;
- нужна **частота** или **самый частый**;
- надо **группировать** по общему ключу;
- задача про **подмассив с суммой `k`** (`prefix_sum` + `dict`);

→ почти всегда правильный ход — пройти массив один раз и накопить нужное в `dict`/`set`/`Counter`/`defaultdict`.


## Скользящее окно (Sliding Window)

Окно фиксированной или переменной длины двигается по массиву. Не пересчитываем сумму с нуля — добавляем правый элемент, вычитаем левый.

Применения: максимальная/минимальная сумма подмассива длины `k`, длиннейшая подстрока без повторов, минимальная подстрока, содержащая все символы.


### Максимальная сумма подмассива длины k


In [ ]:
def max_subarray_sum(arr, k):
    if len(arr) < k:
        return None
    window = sum(arr[:k])
    best = window
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]   # окно сдвинулось вправо
        best = max(best, window)
    return best

max_subarray_sum([2, 1, 5, 1, 3, 2], 3)   # 9 ([5, 1, 3])

### Длиннейшая подстрока без повторяющихся символов


In [ ]:
def longest_unique_substring(s):
    seen = {}
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1     # сдвигаем левую границу
        seen[ch] = right
        best = max(best, right - left + 1)
    return best

longest_unique_substring('abcabcbb')   # 3 ('abc')

## Префиксные суммы (Prefix Sum)

`prefix[i]` = сумма всех элементов от `0` до `i-1`. После этого сумма любого `[l, r]` считается за `O(1)` как `prefix[r+1] - prefix[l]`.

Применения: range sum query, подмассивы с заданной суммой, поиск равных частей.


In [ ]:
def build_prefix(arr):
    prefix = [0]
    for x in arr:
        prefix.append(prefix[-1] + x)
    return prefix

def range_sum(prefix, l, r):
    return prefix[r + 1] - prefix[l]

arr = [3, 1, 4, 1, 5, 9, 2, 6]
p = build_prefix(arr)
range_sum(p, 2, 5)   # 4+1+5+9 = 19

### Подмассив с заданной суммой (через префиксы и хеш)


In [ ]:
def subarray_sum_equals_k(arr, k):
    """Количество подмассивов с суммой = k."""
    count = 0
    prefix_sum = 0
    seen = {0: 1}   # пустой префикс
    for x in arr:
        prefix_sum += x
        # если когда-то была префикс-сумма (prefix_sum - k),
        # значит между ней и текущей позицией сумма = k
        count += seen.get(prefix_sum - k, 0)
        seen[prefix_sum] = seen.get(prefix_sum, 0) + 1
    return count

subarray_sum_equals_k([1, 1, 1], 2)   # 2

## Битовые операции

Полезно знать, если попадётся задача про числа/множества.

| Операция | Что делает |
|---|---|
| `a & b` | AND (битовое И) |
| `a \| b` | OR (битовое ИЛИ) |
| `a ^ b` | XOR (исключающее ИЛИ) |
| `~a` | NOT (инверсия) |
| `a << k` | сдвиг влево (× 2^k) |
| `a >> k` | сдвиг вправо (// 2^k) |


### Полезные трюки


In [ ]:
# Проверить чётность
print(7 & 1)   # 1 — нечётное
print(8 & 1)   # 0 — чётное

# Степень двойки?
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0
print(is_power_of_two(16))   # True
print(is_power_of_two(18))   # False

# Сколько единичных бит
print(bin(13).count('1'))   # 3 (1101)

# Найти единственное непарное число в массиве (все остальные парные)
from functools import reduce
import operator
arr = [1, 2, 3, 2, 1]
print(reduce(operator.xor, arr))   # 3

## Чек-лист для собеседования

1. **Уточнить условия**: размер входа, диапазон значений, отсортирован ли вход, могут ли быть отрицательные.
2. **Привести пример** руками, чтобы убедиться, что понял задачу.
3. **Назвать наивное решение и его сложность** (грубо: `O(n²)`).
4. **Подумать о структурах**: hash map? heap? two pointers? sliding window? стек? DP?
5. **Сначала идея, потом код**. Объяснить план интервьюеру.
6. **Покрыть edge cases**: пустой ввод, один элемент, дубликаты, переполнение.
7. **Назвать итоговую сложность** — по времени **и** по памяти.
